<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.9_Praktikum_Faster_R-CNN.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.9: Praktikum Faster R-CNN (RPN dan RoIAlign)
**Tujuan Pembelajaran:**
1. Membangun modul Region Proposal Network (RPN) di PyTorch.
2. Mengimplementasikan operasi penyelarasan spasial beresolusi tinggi menggunakan `torchvision.ops.RoIAlign`.
3. Mensimulasikan pipeline inferensi detektor dua tahap pada peta fitur konvolusional.

In [ ]:
import torch
import torch.nn as nn
import torchvision.ops as ops
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Definisi Modul Region Proposal Network (RPN)

In [ ]:
class RPNBlock(nn.Module):
    def __init__(self, in_channels=256, num_anchors=9):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.cls_score = nn.Conv2d(in_channels, num_anchors * 2, kernel_size=1)
        self.bbox_pred = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=1)
        
    def forward(self, x):
        t = self.relu(self.conv(x))
        scores = self.cls_score(t)
        deltas = self.bbox_pred(t)
        return scores, deltas

rpn = RPNBlock(in_channels=256, num_anchors=9)
dummy_feat = torch.randn(2, 256, 14, 14)
scores, deltas = rpn(dummy_feat)
print(f"Bentuk Tensor Fitur Input: {dummy_feat.shape}")
print(f"Bentuk Tensor Skor RPN:    {scores.shape} (Batch, 2*9, H, W)")
print(f"Bentuk Tensor Delta BBox:  {deltas.shape} (Batch, 4*9, H, W)")
print("[VALIDASI SUKSES] Modul RPN beroperasi sempurna!")

### Bagian 2: Ekstraksi Fitur Spasial Presisi Menggunakan RoIAlign

In [ ]:
# Inisialisasi RoIAlign: skala 1/16, ukuran output tetap 7x7
roi_align = ops.RoIAlign(output_size=(7, 7), spatial_scale=1.0/16.0, sampling_ratio=2)

# Format RoIs untuk PyTorch ops: [batch_index, x1, y1, x2, y2] dalam koordinat citra asli (misal 224x224)
mock_rois = torch.tensor([
    [0,  32.0,  32.0,  96.0,  96.0],  # RoI 1 pada Batch 0
    [0, 100.0, 100.0, 180.0, 180.0],  # RoI 2 pada Batch 0
    [1,  50.0,  50.0, 120.0, 150.0]   # RoI 3 pada Batch 1
], dtype=torch.float32)

aligned_features = roi_align(dummy_feat, mock_rois)
print(f"Bentuk Tensor Fitur Terpadu RoIAlign: {aligned_features.shape}")
assert aligned_features.shape == (3, 256, 7, 7), "Dimensi RoIAlign salah!"
print("[VALIDASI SUKSES] RoIAlign menghasilkan resolusi tetap 7x7 tanpa kuantisasi integer!")

### Bagian 3: Klasifikasi Akhir Fast R-CNN Head dan Visualisasi Aktivasi

In [ ]:
class FastRCNNHead(nn.Module):
    def __init__(self, in_channels=256, num_classes=3):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels * 7 * 7, 512),
            nn.ReLU(inplace=True)
        )
        self.cls = nn.Linear(512, num_classes)
        self.reg = nn.Linear(512, num_classes * 4)
        
    def forward(self, x):
        feat = self.fc(x)
        return self.cls(feat), self.reg(feat)

head = FastRCNNHead(in_channels=256, num_classes=3)
logits, bbox_refinements = head(aligned_features)
print(f"Logits Kelas Prediksi: {logits.shape} (3 Proposal, 3 Kelas)")
print(f"Regresi Koordinat:     {bbox_refinements.shape} (3 Proposal, 3*4 Offset)")

# Visualisasi nilai aktivasi rata-rata RoIAlign pada salah satu proposal
sample_map = aligned_features[0, 0].detach().numpy()
plt.figure(figsize=(4, 4))
plt.imshow(sample_map, cmap='magma')
plt.colorbar()
plt.title('Representasi Fitur 7x7 Hasil RoIAlign')
plt.tight_layout()
plt.savefig('praktikum_roialign_feature_map_12_9.png', dpi=200)
plt.close()
print("[OK] Visualisasi fitur RoIAlign disimpan sebagai 'praktikum_roialign_feature_map_12_9.png'")